Imports

In [34]:
from dotenv import load_dotenv
import requests
import os
from firecrawl import Firecrawl
from IPython.display import Markdown, display
import json
from datetime import datetime

Archive selection logic

In [ ]:
response = requests.get("https://www.morningbrew.com/api/sanity/issues/archive?end=3000")
print(response.status_code)
print(response.text[:500])

# Doing a simple request still doesn't work

403
<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><meta http-equiv="content-security-policy" content="default-src &#39;none&#39;; script-src &#39;nonce-okqQFzZG2w6yx68aqivCg1&#39; &#39;unsafe-eval&#39; https://challenges.cloudflare.com; script-s


In [ ]:
client = Firecrawl(api_key=api_key)

result = client.scrape(
    "https://www.morningbrew.com/api/sanity/issues/archive?end=3000",
    formats=["markdown"]
)

display(Markdown(result.markdown))

Document(markdown='```json\n[{"_id":"slV5a3vuYndT7PDVh24Aj8","date":"2026-08-13T06:00:00-04:00","previewText":"The Lakers are getting a new owner again...","slug":"what-about-bob","subjectLine":"What about Bob","title":null,"websitePreviewText":null},{"_id":"jv9ddDRH3eco4q8twvDNwj","date":"2026-08-12T06:00:00-04:00","previewText":"The eclipse is boosting Euro tourism...","slug":"solar-windfall","subjectLine":"Solar windfall","title":null,"websitePreviewText":null},{"_id":"f0f5e20f-9ba4-4bb8-9624-d266eb6c348a","date":"2026-08-11T10:00:00.000Z","previewText":"Backlash to Flock cameras grows...","slug":"work-boots-and-subarus","subjectLine":"Work boots and Subarus","title":null,"websitePreviewText":null},{"_id":"jk69j8k0iDJWa9maeCzE2Q","date":"2026-08-10T06:00:00-04:00","previewText":"Buc-ee’s ramps up trademark protection efforts…","slug":"beaver-suit","subjectLine":"Beaver suit","title":null,"websitePreviewText":null},{"_id":"jk69j8k0iDJWa9maeCzBaf","date":"2026-08-09T06:00:00-04:00","p

In [15]:
display(result.markdown)

'```json\n[{"_id":"slV5a3vuYndT7PDVh24Aj8","date":"2026-08-13T06:00:00-04:00","previewText":"The Lakers are getting a new owner again...","slug":"what-about-bob","subjectLine":"What about Bob","title":null,"websitePreviewText":null},{"_id":"jv9ddDRH3eco4q8twvDNwj","date":"2026-08-12T06:00:00-04:00","previewText":"The eclipse is boosting Euro tourism...","slug":"solar-windfall","subjectLine":"Solar windfall","title":null,"websitePreviewText":null},{"_id":"f0f5e20f-9ba4-4bb8-9624-d266eb6c348a","date":"2026-08-11T10:00:00.000Z","previewText":"Backlash to Flock cameras grows...","slug":"work-boots-and-subarus","subjectLine":"Work boots and Subarus","title":null,"websitePreviewText":null},{"_id":"jk69j8k0iDJWa9maeCzE2Q","date":"2026-08-10T06:00:00-04:00","previewText":"Buc-ee’s ramps up trademark protection efforts…","slug":"beaver-suit","subjectLine":"Beaver suit","title":null,"websitePreviewText":null},{"_id":"jk69j8k0iDJWa9maeCzBaf","date":"2026-08-09T06:00:00-04:00","previewText":"A loo

In [16]:
result.markdown.strip("'`json\n")

'[{"_id":"slV5a3vuYndT7PDVh24Aj8","date":"2026-08-13T06:00:00-04:00","previewText":"The Lakers are getting a new owner again...","slug":"what-about-bob","subjectLine":"What about Bob","title":null,"websitePreviewText":null},{"_id":"jv9ddDRH3eco4q8twvDNwj","date":"2026-08-12T06:00:00-04:00","previewText":"The eclipse is boosting Euro tourism...","slug":"solar-windfall","subjectLine":"Solar windfall","title":null,"websitePreviewText":null},{"_id":"f0f5e20f-9ba4-4bb8-9624-d266eb6c348a","date":"2026-08-11T10:00:00.000Z","previewText":"Backlash to Flock cameras grows...","slug":"work-boots-and-subarus","subjectLine":"Work boots and Subarus","title":null,"websitePreviewText":null},{"_id":"jk69j8k0iDJWa9maeCzE2Q","date":"2026-08-10T06:00:00-04:00","previewText":"Buc-ee’s ramps up trademark protection efforts…","slug":"beaver-suit","subjectLine":"Beaver suit","title":null,"websitePreviewText":null},{"_id":"jk69j8k0iDJWa9maeCzBaf","date":"2026-08-09T06:00:00-04:00","previewText":"A look at rais

In [ ]:
full_archive = json.loads(result.markdown.strip("'`json\n"))

In [35]:
date = datetime.strptime((full_archive[0]["date"].split("T"))[0],"%Y-%m-%d").date()
date

datetime.date(2026, 8, 13)

In [39]:
cutoff = date.fromisoformat("2020-10-31")
cutoff

datetime.date(2020, 10, 31)

In [19]:
# Single block

def scrape_archive():
    client = Firecrawl(api_key=api_key)

    result = client.scrape(
        "https://www.morningbrew.com/api/sanity/issues/archive?end=3000",
        formats=["markdown"]
    )

    return json.loads(result.markdown.strip("'`json\n"))

full_archive = scrape_archive()


In [59]:
# Define date cutoffs

first_thursday = date.fromisoformat("2020-11-05")
first_worded_slug = date.fromisoformat("2021-09-30")

valid_weekdays = [3,5] # Thursday, Saturday

def extract_slug(item):
    date = datetime.strptime((item["date"].split("T"))[0],"%Y-%m-%d").date()
    if date.weekday() not in valid_weekdays:
        return None
    elif date.weekday() == 3 and date >= first_thursday:
        if date >= first_worded_slug:
            return item["slug"]
        else:
            return item["slug"].replace("/", "%2F")
    elif date.weekday() == 5:
        return item["slug"].replace("/", "%2F")



In [70]:
extract_slug(full_archive[64])

Everything else

In [4]:
load_dotenv("../.env")

api_key = os.getenv("FIRECRAWL_API_KEY")

print(api_key[:10])

fc-f48a711


In [ ]:
# Parameters (hardcoding for testing)

date = "2019-03-06T10:00:00.000+00:00"
slug = "2019%2F11%2F16"


In [ ]:
# Scrape issue

def scrape_issue(slug):
    client = Firecrawl(api_key=api_key)
    scraped_issue = client.scrape(
        f"https://www.morningbrew.com/issues/{slug}",
        formats=["markdown"]
    )

    return scraped_issue

scraped_issue = scrape_issue(slug)

In [ ]:
# Cut everything before the quiz and break into parts

def cut_before_and_split(scraped_issue):
    # if date <= 2020-10-31
        quiz_section_start_index = scraped_issue.markdown.lower().find("saturday headlines")
    # if date > 2020-10-31
        quiz_section_start_index = scraped_issue.markdown.lower().find("three headlines and a lie")
    quiz_section = scraped_issue.markdown[quiz_section_start_index:]
